# 00 — Dataset and Semantic Segmentation Fundamentals

**Goal:** understand what the model must predict before writing the architecture.

This notebook explores Oxford-IIIT Pet images and their trimap masks. It also verifies the label mapping used by the training code.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
from torchvision.datasets import OxfordIIITPet

dataset = OxfordIIITPet(root="../data", split="trainval",
                        target_types="segmentation", download=True)
print("Number of training/validation images:", len(dataset))

## 1. What is semantic segmentation?

A classifier predicts a category for an entire image. An object detector predicts categories and bounding boxes. A semantic segmentation model predicts a class for every pixel.

For this dataset, the mask is a **trimap** with three semantic labels:
- raw value 1: pet
- raw value 2: background
- raw value 3: boundary

Our code maps these to contiguous class IDs 0, 1 and 2. The mapping is explicit because the model's class channels must correspond to known meanings.

In [ ]:
from src.data import RAW_TO_CLASS, CLASS_NAMES
print("Raw-to-model label mapping:", RAW_TO_CLASS)
print("Class order:", CLASS_NAMES)

## 2. Inspect image and mask pairs

The image is a color photograph. The mask is not a photograph: each integer value is a class identifier. Do not use bilinear interpolation on the mask because it creates intermediate values that are not valid classes.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for row, idx in enumerate([0, 5, 25]):
    image, mask = dataset[idx]
    image_np = np.asarray(image)
    mask_np = np.asarray(mask)
    axes[row, 0].imshow(image_np); axes[row, 0].set_title("Input image")
    axes[row, 1].imshow(mask_np, cmap="viridis", vmin=1, vmax=3); axes[row, 1].set_title("Raw trimap")
    axes[row, 2].imshow(mask_np == 1, cmap="gray"); axes[row, 2].set_title("Pet pixels")
    axes[row, 3].imshow(mask_np == 3, cmap="gray"); axes[row, 3].set_title("Boundary pixels")
    for ax in axes[row]: ax.axis("off")
plt.tight_layout()

## 3. Check label values and class balance

Class balance matters. If one class occupies most pixels, pixel accuracy may look strong even when a small class is poorly predicted. We will therefore report per-class metrics and macro averages.

In [ ]:
from collections import Counter
counts = Counter()
for i in range(min(len(dataset), 100)):
    _, mask = dataset[i]
    counts.update(np.asarray(mask).ravel().tolist())
print("Raw mask pixel counts in first 100 images:", dict(counts))

## 4. Questions to answer in your notes

1. Why is segmentation called dense prediction?
2. What is the difference between semantic and instance segmentation?
3. Why does the pet dataset have a boundary class?
4. Why should a mask be resized with nearest-neighbor interpolation?
5. Why should the test set remain separate until final evaluation?

**Deliverable:** add your own written answers and save a few inspected image/mask examples in your project notes.